In [ ]:
# Import required libraries
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import csv
from torchvision import models
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from sklearn.metrics import precision_recall_fscore_support
from tqdm import tqdm
import unicodedata

from google.colab import drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore')

os.environ["TORCH_HOME"] = '/opt/torch_models'

# Constants
NUM_CLASSES = 100
TARGET_FRAMES = 16  # number of frames per video


BASE_PATH = '/content/drive/MyDrive/AI_PROJECT/sign-language-detector'
DATASET_ZIP = os.path.join(BASE_PATH, 'dataset.zip')
LOCAL_EXTRACT_PATH = '/content/dataset_local'

if not os.path.exists(LOCAL_EXTRACT_PATH) or not os.listdir(LOCAL_EXTRACT_PATH):
    os.makedirs(LOCAL_EXTRACT_PATH, exist_ok=True)
    if not os.path.exists(DATASET_ZIP):
        raise FileNotFoundError(f"❌ Không tìm thấy file zip tại: {DATASET_ZIP}")
    !unzip -q "{DATASET_ZIP}" -d "{LOCAL_EXTRACT_PATH}"
    print(f"✓ Successfully unzip in: {LOCAL_EXTRACT_PATH}")
else:
    print("Data have already existed, skip unzip step.")

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

TRAIN_ROOT = '/content/dataset_local/dataset/train'
LABEL_MAP_PATH = os.path.join(LOCAL_EXTRACT_PATH, 'dataset/label_mapping.pkl')

if os.path.exists(TRAIN_ROOT):
    folders = sorted(os.listdir(TRAIN_ROOT))
    print(f"Total of classes: {len(folders)}")
    print("10 first classes:")
    for f in folders[:10]:
        print(f" - {f}")
else:
    print("Train directory not found.")

In [ ]:
!pip install wandb -q
import wandb
wandb.login()

WANDB_PROJECT = "video-classification-100"
WANDB_ENTITY = "phucduong0308-ho-chi-minh-city-university-of-technology"

In [ ]:
# Read video frames using OpenCV
def read_video(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    cap.release()
    if len(frames) == 0:
        raise ValueError(f"Could not read any frames from {video_path}")
    frames = torch.from_numpy(np.stack(frames, axis=0))
    return frames


# Custom collate function for batching
def collate_fn(batch):
    frames = torch.stack([item['frames'] for item in batch])
    labels = torch.tensor([item['label_idx'] for item in batch])
    label_names = [item['label'] for item in batch]
    return {'frames': frames, 'label_idx': labels, 'label': label_names}

In [ ]:
# Define video dataset
class VideoDataset(Dataset):
    def __init__(self, root_dir, label_to_idx_path, transform=None,
                 mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
                 target_frames=32, is_train = False): #Mới thêm is_train = false
        self.root_dir = root_dir
        self.transform = transform
        self.mean, self.std = mean, std
        self.target_frames = target_frames
        self.instances, self.labels, self.label_idx = [], [], []
        self.is_train = is_train #NEWLY ADDED

        # 1. Đọc và CHUẨN HÓA toàn bộ dictionary từ file pkl
        with open(label_to_idx_path, 'rb') as f:
            raw_mapping = pickle.load(f)

        self.label_mapping = {}
        for k, v in raw_mapping.items():
            clean_key = unicodedata.normalize('NFC', k).strip()
            self.label_mapping[clean_key] = v

        # 2. Duyệt qua các thư mục
        for label_folder in sorted(os.listdir(root_dir))[:NUM_CLASSES]:
            path = os.path.join(root_dir, label_folder)

            if os.path.isdir(path):
                # Chuẩn hóa tên thư mục giống hệt như chuẩn hóa dictionary
                clean_folder = unicodedata.normalize('NFC', label_folder).strip()

                # 3. KIỂM TRA LỖI NGAY TẠI ĐÂY (trước khi vào vòng lặp video)
                if clean_folder not in self.label_mapping:
                    raise KeyError(f"\nLỖI: Không tìm thấy nhãn '{clean_folder}' trong mapping!\n"
                                   f"Các nhãn đang có: {list(self.label_mapping.keys())}")

                # 4. Load video path
                for video_file in os.listdir(path):
                    video_path = os.path.join(path, video_file)
                    self.instances.append(video_path)
                    self.labels.append(label_folder)
                    self.label_idx.append(self.label_mapping[clean_folder])

    # Downsample frames to fixed length
    def _downsample_frames(self, frames):
        num_frames = frames.shape[0]
        if num_frames == self.target_frames:
            return frames
        elif num_frames < self.target_frames:
            pad = self.target_frames - num_frames
            return torch.cat([frames, frames[-1:].repeat(pad, 1, 1, 1)], dim=0)
        else:
            idx = torch.linspace(0, num_frames - 1, self.target_frames).long()
            return frames[idx]

    # Normalize frames with ImageNet stats
    def _normalize(self, frames):
        frames = frames.permute(0, 3, 1, 2).float() / 255.0
        mean = torch.tensor(self.mean).view(1, 3, 1, 1)
        std = torch.tensor(self.std).view(1, 3, 1, 1)
        return (frames - mean) / std

    def __len__(self):
        return len(self.instances)

    def __getitem__(self, idx):
        video_path = self.instances[idx]
        label, label_idx = self.labels[idx], self.label_idx[idx]
        frames = read_video(video_path)

        #NEWLY ADDED
        if self.is_train and self.transform:
          frames = self.transform._speed_augment(frames)

        frames = self._downsample_frames(frames)

        #NEWLY ADDED
        if self.is_train and self.transform:
          frames = self.transform._color_jitter(frames)
          frames = self.transform._random_resized_crop(frames)

        frames = self._normalize(frames)
        return {"frames": frames, "label_idx": label_idx, "label": label}

In [ ]:
from enum import KEEP

import random

class VideoAugmentation:
  def __init__(self,
               crop_scale = (0.85, 1.0),
               brightness = 0.25,
               contrast = 0.25,
               saturation = 0.25,
               speed_range = (0.9, 1.1)):
               self.crop_scale = crop_scale
               self.brightness = brightness
               self.contrast = contrast
               self.saturation = saturation
               self.speed_range = speed_range

  def _speed_augment(self, frames):
      T = frames.shape[0]
      speed = random.uniform(self.speed_range[0], self.speed_range[1])

      new_T = int(T / speed)
      if new_T < 4:
        new_T = 4
      if new_T == T:
        return frames

      indices = torch.linspace(0, T-1, new_T).long()
      indices = torch.clamp(indices, 0, T-1)
      frames = frames[indices]

      return frames

  def _random_resized_crop(self, frames):
      T, H, W, C = frames.shape
      scale = random.uniform(self.crop_scale[0], self.crop_scale[1])
      crop_h, crop_w = int(H* scale), int(W * scale)
      top = random.randint(0, H - crop_h)
      left = random.randint(0, W - crop_w)
      frames = frames[:, top:top+crop_h, left:left+crop_w, :]
      frames = frames.permute(0,3,1,2).float()
      frames = F.interpolate(frames, size=(224,224), mode='bilinear', align_corners=True)
      frames = frames.permute(0,2,3,1)

      return frames.to(torch.uint8)

  def _color_jitter(self, frames):
    brightness_factor = 1 + random.uniform(-self.brightness, self.brightness)
    contrast_factor = 1 + random.uniform(-self.contrast, self.contrast)
    saturation_factor = 1 + random.uniform(-self.saturation, self.saturation)

    frames = frames.float()

    frames = frames * brightness_factor

    mean = frames.mean(dim = (1, 2), keepdim = True)
    frames = (frames - mean) * contrast_factor + mean

    gray = frames.mean(dim = -1, keepdim = True)
    frames = gray + (frames - gray) * saturation_factor

    frames = torch.clamp(frames, 0, 255)

    return frames.to(torch.uint8)


In [ ]:
#FIXED
from torch.utils.data import WeightedRandomSampler
from torch.utils.data.sampler import SubsetRandomSampler

def create_balanced_sampler(dataset, max_oversample_ratio=10):
    if hasattr(dataset, 'dataset'):
        all_labels = [dataset.dataset.label_idx[i] for i in dataset.indices]
    else:
        all_labels = dataset.label_idx

    label_count = np.bincount(all_labels)

    # Tránh division by zero nếu có class không có mẫu nào
    label_count = np.where(label_count == 0, 1, label_count)

    # Clamp weight để tránh oversample minority class quá nhiều lần
    max_count = label_count.max()
    min_count = label_count[label_count > 1].min()

    # Cap ratio: minority class không được sample quá max_oversample_ratio lần majority
    capped_count = np.clip(label_count, min_count / max_oversample_ratio, None)
    weights = 1.0 / capped_count

    sample_weights = torch.FloatTensor([weights[label] for label in all_labels])

    # num_samples = majority class count * num_classes để epoch không bị quá dài
    target_samples = int(max_count * len(label_count))
    target_samples = min(target_samples, len(all_labels) * 2)  # không quá 2x dataset gốc

    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=target_samples,
        replacement=True
    )

    print(f'Balanced sampler: {len(all_labels)} -> {target_samples} samples/epoch')
    print(f'Oversample ratio capped at {max_oversample_ratio}x')
    return sampler


In [ ]:
# Define CRNN model
class CRNN(nn.Module):
    def __init__(self, num_classes=100, hidden_size=256, resnet_pretrained_weights=None):
        super(CRNN, self).__init__()
        resnet = models.resnet50(weights=resnet_pretrained_weights)
        self.cnn = nn.Sequential(*list(resnet.children())[:-2])
        self.feature_dim = 2048
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.rnn = nn.LSTM(self.feature_dim, hidden_size, batch_first=True, dropout=0.3)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        features = self.cnn(x)
        pooled = self.pool(features).view(B, T, self.feature_dim)
        rnn_out, _ = self.rnn(pooled)
        final = rnn_out[:, -1, :]
        return self.fc(final)

In [ ]:
# One training epoch
def train_epoch(model, dataloader, criterion, optimizer, device='cuda', accumulation_steps=4):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    progress = tqdm(dataloader, desc='Training')
    for i, batch in enumerate(progress):
        frames, labels = batch['frames'].to(device), batch['label_idx'].to(device)
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss = loss / accumulation_steps
        loss.backward()
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()
        total_loss += loss.item() * accumulation_steps
        progress.set_postfix({'loss': f'{total_loss / (i+1+1e-9):.4f}'})

    return total_loss / len(dataloader)

#"Do giới hạn phần cứng (VRAM 15GB), mô hình ResNet50 không thể chạy với batch_size = 32 như Baseline ResNet18. Để đảm bảo tính công bằng (fair comparison) trong thực nghiệm, nhóm đã áp dụng kỹ thuật Gradient Accumulation với batch_size = 8 và accumulation_steps = 4. Điều này giúp effective_batch_size duy trì ở mức 32, đảm bảo quá trình cập nhật gradient của ResNet50 tương đồng hoàn toàn với ResNet18."
# Validation
def validate(model, dataloader, criterion, device='cuda'):
    model.eval()
    total_loss, preds, labels_all = 0, [], []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Validation'):
            frames, labels = batch['frames'].to(device), batch['label_idx'].to(device)
            outputs = model(frames)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            preds.extend(predicted.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    precision, recall, f1, _ = precision_recall_fscore_support(labels_all, preds, average='macro', zero_division=0)
    return total_loss / len(dataloader), {'precision': precision*100, 'recall': recall*100, 'f1': f1*100}

In [ ]:
# Full training loop with validation and test evaluation
def train_model(model, train_loader, val_loader,
                num_epochs=10, lr=5e-4, device='cuda', save_path='best_model.pth'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing = 0.1) #FIXED
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)

    best_f1 = 0.0
    for epoch in range(num_epochs):
        print(f"\n===== Epoch {epoch+1}/{num_epochs} =====")
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_metrics = validate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_f1": val_metrics['f1'],
            "val_precision": val_metrics['precision'],
            "val_recall": val_metrics['recall'],
            "learning_rate": optimizer.param_groups[0]['lr']
        })

        print(f"Val F1: {val_metrics['f1']:.2f}% | Precision: {val_metrics['precision']:.2f}% | Recall: {val_metrics['recall']:.2f}%")

        if val_metrics['f1'] > best_f1:
            best_f1 = val_metrics['f1']
            torch.save(model.state_dict(), save_path)
            print(f"✓ Best model saved with F1: {best_f1:.2f}%")
    return model

In [ ]:
run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name="CRNN_ResNet50_dqt",
    config={
        "architecture": "CRNN-ResNet50",
        "backbone": "resnet50",
        "learning_rate": 1e-4,
        "epochs": 15,
        "batch_size": 16,
        "target_frames": 16
    }
)

In [ ]:
# Training
#FIXED
model = CRNN(num_classes=NUM_CLASSES, hidden_size=256,
             resnet_pretrained_weights=models.ResNet50_Weights.IMAGENET1K_V1)

augment = VideoAugmentation(
    crop_scale=(0.85, 1.0),
    brightness=0.25,
    contrast=0.25,
    saturation=0.25,
    speed_range=(0.9, 1.1)
)

# Train dataset CÓ augmentation
full_train = VideoDataset(TRAIN_ROOT, LABEL_MAP_PATH,
                          transform=augment,
                          target_frames=TARGET_FRAMES,
                          is_train=True)   # <- bật aug

# Val dataset KHÔNG augmentation
full_val = VideoDataset(TRAIN_ROOT, LABEL_MAP_PATH,
                        transform=None,
                        target_frames=TARGET_FRAMES,
                        is_train=False)   # <- tắt aug

# Split indices
train_size = int(0.8 * len(full_train))
indices = list(range(len(full_train)))
np.random.seed(42)
np.random.shuffle(indices)
train_indices = indices[:train_size]
val_indices   = indices[train_size:]

train_dataset = torch.utils.data.Subset(full_train, train_indices)
val_dataset   = torch.utils.data.Subset(full_val,   val_indices)   # <- dùng full_val

balanced_sampler = create_balanced_sampler(train_dataset, max_oversample_ratio=10)

train_loader = DataLoader(train_dataset, batch_size=8,
                          collate_fn=collate_fn, num_workers=4,
                          sampler=balanced_sampler)

val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False,
                        collate_fn=collate_fn, num_workers=4)

model = train_model(model, train_loader, val_loader,
                    num_epochs=15, lr=1e-4, device='cuda', save_path='best_model.pth')

In [ ]:
# Evaluate trained model on test set
def evaluate(model, folder_path, label_to_idx_path, output_csv="predictions.csv",
             device='cuda', model_path=None, target_frames=16):
    # Load trained weights if provided
    if model_path:
        model.load_state_dict(torch.load(model_path))
        print(f"Loaded model from {model_path}")

    model = model.to(device)
    model.eval()

    # Load label mapping
    with open(label_to_idx_path, 'rb') as f:
        label_mapping = pickle.load(f)
    idx_to_label = {v: k for k, v in label_mapping.items()}

    # Collect video files
    video_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))])
    print(f"Found {len(video_files)} videos in '{folder_path}'")

    predictions = []

    dataset = VideoDataset(
                  root_dir=folder_path,
                  label_to_idx_path=label_to_idx_path,
                  target_frames=target_frames
              )

    with torch.no_grad():
        for video_file in tqdm(video_files, desc="Predicting"):
            video_path = os.path.join(folder_path, video_file)
            try:
                # Read and preprocess video
                frames = read_video(video_path)
                frames = dataset._downsample_frames(frames)
                frames = dataset._normalize(frames)
                frames = frames.unsqueeze(0).to(device)  # (1, T, C, H, W)

                # Predict
                outputs = model(frames)
                _, predicted = outputs.max(1)
                label_idx = predicted.item()
                label_name = idx_to_label[label_idx]

                predictions.append((video_file, label_name))
            except Exception as e:
                print(f"Error processing {video_file}: {e}")

    # Save to CSV
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['video_name', 'label'])
        writer.writerows(predictions)

    print(f"\nPredictions saved to '{output_csv}'")
    print(f"Total videos processed: {len(predictions)}")

In [ ]:
# Export public result
evaluate(
    model=model,
    folder_path=os.path.join(LOCAL_EXTRACT_PATH, "dataset/public_test"),
    label_to_idx_path=LABEL_MAP_PATH,
    model_path="best_model.pth",
    output_csv="public_submission.csv",
    device="cuda",
    target_frames=16
)

import zipfile

with zipfile.ZipFile("public_submission.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("public_submission.csv")
        print("Created file public_submission.zip successfully.")

In [ ]:
# Export private result
evaluate(
    model=model,
    folder_path=os.path.join(LOCAL_EXTRACT_PATH, "dataset/private_test"),
    label_to_idx_path=LABEL_MAP_PATH,
    model_path="best_model.pth",
    output_csv="private_submission.csv",
    device="cuda",
    target_frames=16
)

import zipfile

with zipfile.ZipFile("private_test.zip", 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write("private_submission.csv")
        print("Created file private_submission.zip successfully.")

In [ ]:
!cp /content/best_model.pth "/content/drive/MyDrive/AI_PROJECT/sign-language-detector/best_model_resnet50.pth"

!cp /content/public_submission.csv "/content/drive/MyDrive/AI_PROJECT/sign-language-detector/"
!cp /content/public_submission.zip "/content/drive/MyDrive/AI_PROJECT/sign-language-detector/"

!cp /content/private_submission.csv "/content/drive/MyDrive/AI_PROJECT/sign-language-detector/"
!cp /content/private_test.zip "/content/drive/MyDrive/AI_PROJECT/sign-language-detector/"
